# Assignment 5 — Smoothing Techniques for N-Gram Language Models

Use the **4 models from Assignment 4** (Unigram, Bigram, Trigram, Quadrigram) and evaluate them on the same **training, development, and test split**.

Smoothing techniques implemented:
1. Interpolated Smoothing — Bigram, Trigram, Quadrigram
2. Good-Turing Smoothing
3. Katz Backoff Smoothing
4. Stupid Backoff Smoothing
5. Kneser-Ney Smoothing

The counts are loaded from Assignment 4, so the four n-gram models do not need to be trained again.

In [1]:
import pandas as pd
import pickle
import math
from collections import Counter, defaultdict

In [2]:
# Load the Marathi tokenized dataset
data = pd.read_parquet("mar_Deva_tokenized.parquet")

# Same split as Assignment 4
data = data.sample(frac=1, random_state=42).reset_index(drop=True)

test = data.iloc[-1000:]
dev = data.iloc[-2000:-1000]
train = data.iloc[:-2000]

print("Training:", len(train))
print("Development:", len(dev))
print("Test:", len(test))

Training: 2001060
Development: 1000
Test: 1000


In [3]:
def prepare_sentence(sentence, order=2):
    # Add order-1 start tokens and one end token
    return ["<s>"] * (order - 1) + list(sentence) + ["</s>"]

In [4]:
# Load all counts created in Assignment 4

with open("../Assignment 4/unigram_counts.pkl", "rb") as f:
    unigram_counts = pickle.load(f)

with open("../Assignment 4/bigram_counts.pkl", "rb") as f:
    bigram_counts = pickle.load(f)

with open("../Assignment 4/bigram_context_counts.pkl", "rb") as f:
    bigram_context_counts = pickle.load(f)

with open("../Assignment 4/trigram_counts.pkl", "rb") as f:
    trigram_counts = pickle.load(f)

with open("../Assignment 4/trigram_context_counts.pkl", "rb") as f:
    trigram_context_counts = pickle.load(f)

with open("../Assignment 4/quadrigram_counts.pkl", "rb") as f:
    quadrigram_counts = pickle.load(f)

with open("../Assignment 4/quadrigram_context_counts.pkl", "rb") as f:
    quadrigram_context_counts = pickle.load(f)

with open("../Assignment 4/vocabulary.pkl", "rb") as f:
    vocabulary = pickle.load(f)

V = len(vocabulary)

print("Counts and vocabulary loaded successfully.")
print("Vocabulary size:", V)

Counts and vocabulary loaded successfully.
Vocabulary size: 876183


## 1. Interpolated Smoothing

Linear interpolation combines lower-order and higher-order probabilities. The weights below are chosen so that the higher-order model gets more weight while lower-order models still provide probability for unseen n-grams.

In [5]:
# Maximum-likelihood probabilities used by interpolation

total_words = sum(unigram_counts.values())

def unigram_mle(word):
    return unigram_counts[word] / total_words if total_words > 0 else 0

def bigram_mle(word1, word2):
    context = bigram_context_counts[word1]
    return bigram_counts[(word1, word2)] / context if context > 0 else 0

def trigram_mle(word1, word2, word3):
    context = trigram_context_counts[(word1, word2)]
    return trigram_counts[(word1, word2, word3)] / context if context > 0 else 0

def quadrigram_mle(word1, word2, word3, word4):
    context = quadrigram_context_counts[(word1, word2, word3)]
    return quadrigram_counts[(word1, word2, word3, word4)] / context if context > 0 else 0

In [6]:
# Interpolation weights
# Bigram:    unigram + bigram
# Trigram:   unigram + bigram + trigram
# Quadrigram: unigram + bigram + trigram + quadrigram

LAMBDA_BIGRAM = [0.4, 0.6]
LAMBDA_TRIGRAM = [0.2, 0.3, 0.5]
LAMBDA_QUADRIGRAM = [0.1, 0.2, 0.3, 0.4]

def bigram_probability_interpolated(word1, word2):
    return (
        LAMBDA_BIGRAM[0] * unigram_mle(word2)
        + LAMBDA_BIGRAM[1] * bigram_mle(word1, word2)
    )

def trigram_probability_interpolated(word1, word2, word3):
    return (
        LAMBDA_TRIGRAM[0] * unigram_mle(word3)
        + LAMBDA_TRIGRAM[1] * bigram_mle(word2, word3)
        + LAMBDA_TRIGRAM[2] * trigram_mle(word1, word2, word3)
    )

def quadrigram_probability_interpolated(word1, word2, word3, word4):
    return (
        LAMBDA_QUADRIGRAM[0] * unigram_mle(word4)
        + LAMBDA_QUADRIGRAM[1] * bigram_mle(word3, word4)
        + LAMBDA_QUADRIGRAM[2] * trigram_mle(word2, word3, word4)
        + LAMBDA_QUADRIGRAM[3] * quadrigram_mle(word1, word2, word3, word4)
    )

## 2. Good-Turing Smoothing

In [7]:
# Good-Turing adjusted counts

def good_turing_adjusted_count(counts, c, max_count=5):
    if c == 0:
        return 0

    count_of_counts = Counter(counts.values())
    Nc = count_of_counts.get(c, 0)
    Nc1 = count_of_counts.get(c + 1, 0)

    # For reliable higher counts, keep the original count.
    if c > max_count or Nc == 0 or Nc1 == 0:
        return c

    return (c + 1) * Nc1 / Nc

def good_turing_probability_unigram(word):
    N = sum(unigram_counts.values())
    c = unigram_counts[word]

    if c > 0:
        c_star = good_turing_adjusted_count(unigram_counts, c)
        return c_star / N

    # For unseen/OOV words, use the Good-Turing unseen mass.
    count_of_counts = Counter(unigram_counts.values())
    N1 = count_of_counts.get(1, 0)

    # The Assignment 4 vocabulary contains observed words, so keep one
    # unseen bucket for OOV evaluation.
    N0 = max(1, 1)
    return (N1 / N) / N0 if N > 0 else 0

def good_turing_probability_bigram(word1, word2):
    c = bigram_counts[(word1, word2)]
    context = bigram_context_counts[word1]

    if context == 0:
        return good_turing_probability_unigram(word2)

    if c > 0:
        c_star = good_turing_adjusted_count(bigram_counts, c)
        return c_star / context

    count_of_counts = Counter(bigram_counts.values())
    N1 = count_of_counts.get(1, 0)
    N0 = max(1, V * V - len(bigram_counts))
    return (N1 / N0) / context

def good_turing_probability_trigram(word1, word2, word3):
    c = trigram_counts[(word1, word2, word3)]
    context = trigram_context_counts[(word1, word2)]

    if context == 0:
        return good_turing_probability_bigram(word2, word3)

    if c > 0:
        c_star = good_turing_adjusted_count(trigram_counts, c)
        return c_star / context

    count_of_counts = Counter(trigram_counts.values())
    N1 = count_of_counts.get(1, 0)
    N0 = max(1, V ** 3 - len(trigram_counts))
    return (N1 / N0) / context

def good_turing_probability_quadrigram(word1, word2, word3, word4):
    c = quadrigram_counts[(word1, word2, word3, word4)]
    context = quadrigram_context_counts[(word1, word2, word3)]

    if context == 0:
        return good_turing_probability_trigram(word2, word3, word4)

    if c > 0:
        c_star = good_turing_adjusted_count(quadrigram_counts, c)
        return c_star / context

    count_of_counts = Counter(quadrigram_counts.values())
    N1 = count_of_counts.get(1, 0)
    N0 = max(1, V ** 4 - len(quadrigram_counts))
    return (N1 / N0) / context

## 3. Katz Backoff Smoothing

Katz uses Good-Turing adjusted counts for seen n-grams. For unseen n-grams it backs off to a lower-order model using a context-specific backoff factor.

In [8]:
def gt_seen_probability(count, context_count, counts):
    if context_count == 0:
        return 0
    c_star = good_turing_adjusted_count(counts, count)
    return c_star / context_count

# Store observed continuations for each context.
bigram_words_by_context = defaultdict(list)
for (w1, w2) in bigram_counts:
    bigram_words_by_context[w1].append(w2)

trigram_words_by_context = defaultdict(list)
for (w1, w2, w3) in trigram_counts:
    trigram_words_by_context[(w1, w2)].append(w3)

quadrigram_words_by_context = defaultdict(list)
for (w1, w2, w3, w4) in quadrigram_counts:
    quadrigram_words_by_context[(w1, w2, w3)].append(w4)

In [ ]:
# Katz unigram probability
# Normalize the Good-Turing unigram scores over the observed vocabulary
# plus one OOV bucket.

gt_unigram_raw = {}

for word in vocabulary:
    c = unigram_counts[word]
    if c > 0:
        gt_unigram_raw[word] = good_turing_adjusted_count(
            unigram_counts, c
        ) / total_words
    else:
        gt_unigram_raw[word] = 0

gt_unigram_raw["<UNK>"] = good_turing_probability_unigram("<UNK>")

gt_unigram_normalizer = sum(gt_unigram_raw.values())

def katz_unigram_probability(word):
    if word in gt_unigram_raw:
        return gt_unigram_raw[word] / gt_unigram_normalizer

    return gt_unigram_raw["<UNK>"] / gt_unigram_normalizer

In [ ]:
# Build Katz backoff parameters for bigram model

katz_bigram_scale = {}
katz_bigram_alpha = {}

for context, words in bigram_words_by_context.items():
    C = bigram_context_counts[context]

    adjusted = [
        good_turing_adjusted_count(bigram_counts[(context, word)],
                                   bigram_counts[(context, word)])
        for word in words
    ]

    adjusted_total = sum(adjusted)
    scale = min(1.0, C / adjusted_total) if adjusted_total > 0 else 1.0
    katz_bigram_scale[context] = scale

    seen_mass = sum(
        scale * good_turing_adjusted_count(
            bigram_counts[(context, word)],
            bigram_counts[(context, word)]
        ) / C
        for word in words
    )

    lower_seen_mass = sum(katz_unigram_probability(word) for word in words)
    denominator = 1.0 - lower_seen_mass

    if denominator > 0:
        katz_bigram_alpha[context] = max(0.0, (1.0 - seen_mass) / denominator)
    else:
        katz_bigram_alpha[context] = 0.0

def katz_bigram_probability(word1, word2):
    c = bigram_counts[(word1, word2)]

    if c > 0 and word1 in bigram_context_counts:
        C = bigram_context_counts[word1]
        c_star = good_turing_adjusted_count(bigram_counts, c)
        scale = katz_bigram_scale.get(word1, 1.0)
        return scale * c_star / C

    alpha = katz_bigram_alpha.get(word1, 1.0)
    return alpha * katz_unigram_probability(word2)

In [ ]:
# Build Katz backoff parameters for trigram model

katz_trigram_scale = {}
katz_trigram_alpha = {}

for context, words in trigram_words_by_context.items():
    C = trigram_context_counts[context]

    adjusted = [
        good_turing_adjusted_count(
            trigram_counts[(context[0], context[1], word)],
            trigram_counts[(context[0], context[1], word)]
        )
        for word in words
    ]

    adjusted_total = sum(adjusted)
    scale = min(1.0, C / adjusted_total) if adjusted_total > 0 else 1.0
    katz_trigram_scale[context] = scale

    seen_mass = sum(
        scale * good_turing_adjusted_count(
            trigram_counts[(context[0], context[1], word)],
            trigram_counts[(context[0], context[1], word)]
        ) / C
        for word in words
    )

    lower_seen_mass = sum(
        katz_bigram_probability(context[1], word)
        for word in words
    )

    denominator = 1.0 - lower_seen_mass

    if denominator > 0:
        katz_trigram_alpha[context] = max(
            0.0, (1.0 - seen_mass) / denominator
        )
    else:
        katz_trigram_alpha[context] = 0.0

def katz_trigram_probability(word1, word2, word3):
    c = trigram_counts[(word1, word2, word3)]
    context = (word1, word2)

    if c > 0 and context in trigram_context_counts:
        C = trigram_context_counts[context]
        c_star = good_turing_adjusted_count(trigram_counts, c)
        scale = katz_trigram_scale.get(context, 1.0)
        return scale * c_star / C

    alpha = katz_trigram_alpha.get(context, 1.0)
    return alpha * katz_bigram_probability(word2, word3)

In [ ]:
# Build Katz backoff parameters for quadrigram model

katz_quadrigram_scale = {}
katz_quadrigram_alpha = {}

for context, words in quadrigram_words_by_context.items():
    C = quadrigram_context_counts[context]

    adjusted = [
        good_turing_adjusted_count(
            quadrigram_counts[(context[0], context[1], context[2], word)],
            quadrigram_counts[(context[0], context[1], context[2], word)]
        )
        for word in words
    ]

    adjusted_total = sum(adjusted)
    scale = min(1.0, C / adjusted_total) if adjusted_total > 0 else 1.0
    katz_quadrigram_scale[context] = scale

    seen_mass = sum(
        scale * good_turing_adjusted_count(
            quadrigram_counts[(context[0], context[1], context[2], word)],
            quadrigram_counts[(context[0], context[1], context[2], word)]
        ) / C
        for word in words
    )

    lower_seen_mass = sum(
        katz_trigram_probability(context[1], context[2], word)
        for word in words
    )

    denominator = 1.0 - lower_seen_mass

    if denominator > 0:
        katz_quadrigram_alpha[context] = max(
            0.0, (1.0 - seen_mass) / denominator
        )
    else:
        katz_quadrigram_alpha[context] = 0.0

def katz_quadrigram_probability(word1, word2, word3, word4):
    c = quadrigram_counts[(word1, word2, word3, word4)]
    context = (word1, word2, word3)

    if c > 0 and context in quadrigram_context_counts:
        C = quadrigram_context_counts[context]
        c_star = good_turing_adjusted_count(quadrigram_counts, c)
        scale = katz_quadrigram_scale.get(context, 1.0)
        return scale * c_star / C

    alpha = katz_quadrigram_alpha.get(context, 1.0)
    return alpha * katz_trigram_probability(word2, word3, word4)

## 4. Stupid Backoff Smoothing

Stupid Backoff uses the higher-order maximum-likelihood probability when an n-gram is seen. If it is unseen, it backs off to the lower-order probability multiplied by **α = 0.4**.

In [ ]:
STUPID_BACKOFF_ALPHA = 0.4

def stupid_backoff_unigram(word):
    return unigram_mle(word)

def stupid_backoff_bigram(word1, word2):
    c = bigram_counts[(word1, word2)]
    context = bigram_context_counts[word1]

    if c > 0 and context > 0:
        return c / context

    return STUPID_BACKOFF_ALPHA * stupid_backoff_unigram(word2)

def stupid_backoff_trigram(word1, word2, word3):
    c = trigram_counts[(word1, word2, word3)]
    context = trigram_context_counts[(word1, word2)]

    if c > 0 and context > 0:
        return c / context

    return STUPID_BACKOFF_ALPHA * stupid_backoff_bigram(word2, word3)

def stupid_backoff_quadrigram(word1, word2, word3, word4):
    c = quadrigram_counts[(word1, word2, word3, word4)]
    context = quadrigram_context_counts[(word1, word2, word3)]

    if c > 0 and context > 0:
        return c / context

    return STUPID_BACKOFF_ALPHA * stupid_backoff_trigram(word2, word3, word4)

## 5. Kneser-Ney Smoothing

This implementation uses **Interpolated Kneser-Ney** with absolute discounting and **D = 0.75**. Lower-order probabilities are based on continuation counts rather than ordinary unigram frequency.

In [ ]:
KNN_D = 0.75

# Continuation counts for words:
# number of distinct words that occur immediately before each word.

preceding_words = defaultdict(set)

for (w1, w2) in bigram_counts:
    preceding_words[w2].add(w1)

total_distinct_bigrams = len(bigram_counts)

def kneser_ney_unigram(word):
    if total_distinct_bigrams == 0:
        return 0
    return len(preceding_words[word]) / total_distinct_bigrams

def kneser_ney_bigram(word1, word2):
    C = bigram_context_counts[word1]

    if C == 0:
        return kneser_ney_unigram(word2)

    c = bigram_counts[(word1, word2)]
    unique_continuations = len(bigram_words_by_context[word1])

    lambda_value = KNN_D * unique_continuations / C

    first_part = max(c - KNN_D, 0) / C
    backoff_part = lambda_value * kneser_ney_unigram(word2)

    return first_part + backoff_part

In [ ]:
# Kneser-Ney trigram

def kneser_ney_trigram(word1, word2, word3):
    context = (word1, word2)
    C = trigram_context_counts[context]

    if C == 0:
        return kneser_ney_bigram(word2, word3)

    c = trigram_counts[(word1, word2, word3)]
    unique_continuations = len(trigram_words_by_context[context])

    lambda_value = KNN_D * unique_continuations / C

    first_part = max(c - KNN_D, 0) / C
    backoff_part = lambda_value * kneser_ney_bigram(word2, word3)

    return first_part + backoff_part

In [ ]:
# Kneser-Ney quadrigram

def kneser_ney_quadrigram(word1, word2, word3, word4):
    context = (word1, word2, word3)
    C = quadrigram_context_counts[context]

    if C == 0:
        return kneser_ney_trigram(word2, word3, word4)

    c = quadrigram_counts[(word1, word2, word3, word4)]
    unique_continuations = len(quadrigram_words_by_context[context])

    lambda_value = KNN_D * unique_continuations / C

    first_part = max(c - KNN_D, 0) / C
    backoff_part = lambda_value * kneser_ney_trigram(word2, word3, word4)

    return first_part + backoff_part

## Evaluation — Perplexity on Development and Test Sets

In [ ]:
def evaluate_model(data, order, probability_function):
    total_log_probability = 0.0
    total_words = 0

    for sentence in data["words"]:
        words = prepare_sentence(sentence, order)

        for i in range(order - 1, len(words)):
            ngram = words[i - order + 1:i + 1]
            probability = probability_function(*ngram)

            # Avoid log(0) for a model that produces an invalid zero score.
            probability = max(probability, 1e-300)

            total_log_probability += math.log(probability)
            total_words += 1

    return math.exp(-total_log_probability / total_words)

In [ ]:
# Probability functions for all 4 models

model_functions = {
    "Unigram": {
        "Good-Turing": good_turing_probability_unigram,
        "Katz Backoff": katz_unigram_probability,
        "Stupid Backoff": stupid_backoff_unigram,
        "Kneser-Ney": kneser_ney_unigram,
    },
    "Bigram": {
        "Interpolated": bigram_probability_interpolated,
        "Good-Turing": good_turing_probability_bigram,
        "Katz Backoff": katz_bigram_probability,
        "Stupid Backoff": stupid_backoff_bigram,
        "Kneser-Ney": kneser_ney_bigram,
    },
    "Trigram": {
        "Interpolated": trigram_probability_interpolated,
        "Good-Turing": good_turing_probability_trigram,
        "Katz Backoff": katz_trigram_probability,
        "Stupid Backoff": stupid_backoff_trigram,
        "Kneser-Ney": kneser_ney_trigram,
    },
    "Quadrigram": {
        "Interpolated": quadrigram_probability_interpolated,
        "Good-Turing": good_turing_probability_quadrigram,
        "Katz Backoff": katz_quadrigram_probability,
        "Stupid Backoff": stupid_backoff_quadrigram,
        "Kneser-Ney": kneser_ney_quadrigram,
    }
}

orders = {
    "Unigram": 1,
    "Bigram": 2,
    "Trigram": 3,
    "Quadrigram": 4
}

In [ ]:
# Calculate Development and Test perplexity for every required combination

dev_results = {}
test_results = {}

for model, functions in model_functions.items():
    order = orders[model]

    for smoothing, probability_function in functions.items():
        key = f"{model} - {smoothing}"

        dev_results[key] = evaluate_model(
            dev, order, probability_function
        )

        test_results[key] = evaluate_model(
            test, order, probability_function
        )

In [ ]:
print("DEVELOPMENT SET RESULTS")
print("=" * 70)

for key, value in dev_results.items():
    print(f"{key:40s} Perplexity: {value:.4f}")

print("\nTEST SET RESULTS")
print("=" * 70)

for key, value in test_results.items():
    print(f"{key:40s} Perplexity: {value:.4f}")

In [ ]:
# Results in table form

results_table = []

for key in dev_results:
    model, smoothing = key.split(" - ", 1)

    results_table.append({
        "Model": model,
        "Smoothing": smoothing,
        "Development Perplexity": dev_results[key],
        "Test Perplexity": test_results[key]
    })

results_df = pd.DataFrame(results_table)

print(results_df.to_string(index=False))

### Note

- **Interpolated smoothing** is applied to Bigram, Trigram, and Quadrigram models as specified.
- Good-Turing, Katz Backoff, Stupid Backoff, and Kneser-Ney are implemented for the available n-gram models.
- Stupid Backoff produces a **backoff score rather than a normalized probability**, so its reported perplexity should be interpreted as a comparative backoff-score evaluation.
- Run the notebook from the same directory structure used in Assignment 4 so that the pickle files can be loaded from `../Assignment 4/`.